In [1]:
#|default_exp dataset

In [2]:
#| export
from pathlib import Path
import numpy as np
import torch
from typing import Optional, Callable, List
import copy
import re
import os
import glob
import os.path as osp
import numpy as np
import torch
from torch_geometric.data import (InMemoryDataset, Data, download_url,
                                  extract_tar, extract_zip, DataLoader)
from torch_geometric.utils import remove_isolated_nodes
from torch_sparse import SparseTensor
from tqdm import tqdm 
import random
from torch_geometric.data import Batch
import pandas as pd

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
import torch_geometric
torch_geometric.__version__
torch.version.cuda

'11.8'

In [4]:
#| export
def good_luck():
    return True

In [5]:
class CONFIG:
    path = Path('../data/raw/npz_all/npz/layout')
    
    
def get_shape_info(fn):
    dictionary = dict(np.load(fn))
    # Find the maximum key length for alignment
    max_key_length = max(len(key) for key in dictionary.keys())
    res = {}
    for key, value in dictionary.items():
        if isinstance(value, np.ndarray):
            # Use the determined length to format the output
            res[key]=value.shape
            
    res['search'] = Path(fn).parent.parent.stem
    res['complier'] = Path(fn).parent.parent.parent.stem
    res['data_split'] = Path(fn).parent.stem
    res['path'] = Path(fn)
    return pd.DataFrame(pd.Series(res)).T
            
    

In [6]:
list(CONFIG.path.iterdir())

[PosixPath('../data/raw/npz_all/npz/layout/xla'),
 PosixPath('../data/raw/npz_all/npz/layout/nlp'),
 PosixPath('../data/raw/npz_all/npz/layout/processed')]

In [7]:
fns =  glob.glob('../data/raw/npz_all/npz/layout/xla/default/**/*.npz', recursive=True)
len(fns)

76

In [8]:
# res = pd.concat([get_shape_info(i) for i in tqdm(fns)],)
# res.reset_index(drop=True, inplace=True)

In [9]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()

In [10]:
#  parts_cnt = 0
# thres = 1000
# edge_index = torch.tensor(np_file["edge_index"].T)
# runtime = torch.tensor(np_file["config_runtime"])
# op = torch.tensor(np_file["node_feat"])
# op_code = torch.tensor(np_file["node_opcode"])
# config_feats = torch.tensor(np_file["node_config_feat"])
# config_feats = config_feats.view(-1, config_feats.shape[-1])
# config_idx = torch.tensor(np_file["node_config_ids"])
# num_config = torch.tensor(np_file["node_config_feat"].shape[0])
# num_config_idx = torch.tensor(np_file["node_config_feat"].shape[1])
# num_nodes = torch.tensor(np_file["node_feat"].shape[0])
# num_parts = num_nodes // thres + 1
# interval = num_nodes // num_parts
# partptr = torch.arange(0, num_nodes, interval+1)
# if partptr[-1] != num_nodes:
#        partptr = torch.cat([partptr, torch.tensor([num_nodes])])
# data = Data(edge_index=edge_index, op_feats=op, op_code=op_code, config_feats=config_feats, config_idx=config_idx,
#                                 num_config=num_config, num_config_idx=num_config_idx, y=runtime, num_nodes=num_nodes, partptr=partptr, partition_idx = parts_cnt)

In [11]:
class TPUGraphs(InMemoryDataset):

    def __init__(self, root: str, thres: int = 1000,
                 transform: Optional[Callable] = None,
                 pre_transform: Optional[Callable] = None,
                 pre_filter: Optional[Callable] = None,
                 source: str = 'xla',  # 'nlp' or 'xla'
                 search: str = 'default'  # 'random' or 'default'
                ):
        assert source in ('nlp', 'xla')
        assert search in ('random', 'default')
        self.thres = thres
        self.source = source
        self.search = search
        super().__init__(root, transform, pre_transform, pre_filter)
        self.data, self.slices = torch.load(self.processed_paths[0])
        op_feats_mean = torch.mean(self.data.op_feats, dim=0, keepdim=True)
        op_feats_std = torch.std(self.data.op_feats, dim=0, keepdim=True)
        op_feats_std[op_feats_std < 1e-6] = 1
        self.data.op_feats = (self.data.op_feats - op_feats_mean) / op_feats_std
        
    @property
    def raw_file_names(self) -> List[str]:
        return [f'npz_all/npz/layout/{self.source}/{self.search}']

    @property
    def processed_file_names(self) -> List[str]:
        return ['data_segment_{}.pt'.format(self.thres), 'split_dict_segment_{}.pt'.format(self.thres)]


    def process(self):
        data_list = []
        split_names = ['train', 'valid', 'test']
        split_dict = {'train': [], 'valid': [], 'test': []}
        graphs_cnt = 0
        parts_cnt = 0
        for raw_path in self.raw_paths:
            for split_name in tqdm(split_names):
                filenames = glob.glob(osp.join(os.path.join(raw_path, split_name), '*.npz'))
                for filename in filenames:
                    split_dict[split_name].append(graphs_cnt)
                    np_file = dict(np.load(filename))
                    if "edge_index" not in np_file:
                      print('error in', filename)
                    edge_index = torch.tensor(np_file["edge_index"].T)
                    runtime = torch.tensor(np_file["config_runtime"])
                    op = torch.tensor(np_file["node_feat"])
                    op_code = torch.tensor(np_file["node_opcode"])
                    config_feats = torch.tensor(np_file["node_config_feat"])
                    config_feats = config_feats.view(-1, config_feats.shape[-1])
                    config_idx = torch.tensor(np_file["node_config_ids"])
                    num_config = torch.tensor(np_file["node_config_feat"].shape[0])
                    num_config_idx = torch.tensor(np_file["node_config_feat"].shape[1])
                    num_nodes = torch.tensor(np_file["node_feat"].shape[0])
                    num_parts = num_nodes // self.thres + 1
                    interval = num_nodes // num_parts
                    partptr = torch.arange(0, num_nodes, interval+1)
                    if partptr[-1] != num_nodes:
                        partptr = torch.cat([partptr, torch.tensor([num_nodes])])
                    data = Data(edge_index=edge_index, op_feats=op, op_code=op_code, config_feats=config_feats, config_idx=config_idx,
                                num_config=num_config, num_config_idx=num_config_idx, y=runtime, num_nodes=num_nodes, partptr=partptr, partition_idx = parts_cnt)
                    data_list.append(data)
                    graphs_cnt += 1
                    parts_cnt += num_parts * num_config
            torch.save(self.collate(data_list), self.processed_paths[0])
            torch.save(split_dict, self.processed_paths[1])
    def get_idx_split(self):
        return torch.load(self.processed_paths[1])

In [12]:
def preprocess_batch(batch, model, num_sample_configs):
    
    batch_list = batch.to_data_list()
    processed_batch_list = []
    for g in batch_list:
        sample_idx = torch.randint(0, g.num_config.item(), (num_sample_configs,))
        g.y = g.y[sample_idx]
        g.config_feats = g.config_feats.view(g.num_config, g.num_config_idx, -1)[sample_idx, ...]
        g.config_feats = g.config_feats.transpose(0,1)
        g.config_feats_full = torch.zeros((g.num_nodes, num_sample_configs, g.config_feats.shape[-1]), device=g.config_feats.device)
        g.config_feats_full[g.config_idx, ...] += g.config_feats
        g.adj = SparseTensor(row=g.edge_index[0], col=g.edge_index[1], sparse_sizes=(g.num_nodes, g.num_nodes))
        processed_batch_list.append(g)
    return Batch.from_data_list(processed_batch_list), sample_idx
        

In [13]:
ds = TPUGraphs(root= '../data/')

/opt/conda/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:157: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [21]:
dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)

/opt/conda/lib/python3.10/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [25]:
batch_full = next(iter(dl))

In [26]:
batch_full

DataBatch(edge_index=[2, 73881], y=[1960], op_feats=[43615, 140], op_code=[43615], config_feats=[2269680, 18], config_idx=[1158], num_config=[1], num_config_idx=[1], partptr=[45], partition_idx=[1], num_nodes=43615, batch=[43615], ptr=[2])

In [32]:
batch, sampled_idx =  preprocess_batch(batch_full, None, 10)
true = batch.y
batch_list = batch.to_data_list()

In [41]:
batch_list

[Data(edge_index=[2, 73881], y=[10], op_feats=[43615, 140], op_code=[43615], config_feats=[1158, 10, 18], config_idx=[1158], num_config=[1], num_config_idx=[1], partptr=[45], partition_idx=[1], config_feats_full=[43615, 10, 18], adj=[43615, 43615, nnz=73881], num_nodes=43615)]

In [36]:
batch
i = 0

In [39]:
num_parts = len(batch_list[i].partptr) - 1
segment_to_train = np.random.randint(num_parts)

In [40]:
segment_to_train

32

In [1]:
import gc
gc.collect()

65

In [ ]:
sample_idx = torch.randint(0, ds.num_config[0], (100,))
ds[0].y[sample_idx]

In [ ]:
def preprocess_batch(batch, model, num_sample_configs):
    
    batch_list = batch.to_data_list()
    processed_batch_list = []
    for g in batch_list:
        sample_idx = torch.randint(0, g.num_config.item(), (num_sample_configs,))
        g.y = g.y[sample_idx]
        g.config_feats = g.config_feats.view(g.num_config, g.num_config_idx, -1)[sample_idx, ...]
        g.config_feats = g.config_feats.transpose(0,1)
        g.config_feats_full = torch.zeros((g.num_nodes, num_sample_configs, g.config_feats.shape[-1]), device=g.config_feats.device)
        g.config_feats_full[g.config_idx, ...] += g.config_feats
        g.adj = SparseTensor(row=g.edge_index[0], col=g.edge_index[1], sparse_sizes=(g.num_nodes, g.num_nodes))
        processed_batch_list.append(g)
    return Batch.from_data_list(processed_batch_list)

In [ ]:
sample_idx = torch.randint(0, g.num_config.item(), (num_sample_configs,))

In [ ]:
glob.glob(osp.join(os.path.join(ds.raw_paths[0], 'test'), '*.npz'))

In [ ]:
fns =  glob.glob('../data/raw/npz_all/npz/layout/**/*.npz', recursive=True)
len(fns)

In [ ]:
np_file = dict(np.load(random.choice(fns)))
print_shapes(np_file)

In [ ]:
from datasets import load_dataset

# There is only one split on the hub
dataset = load_dataset("OGB/ogbg-molhiv")

#dataset = dataset.shuffle(seed=0)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# We want to plot the first train graph
graph = dataset["train"][0]

edges = graph["edge_index"]
num_edges = len(edges[0])
num_nodes = graph["num_nodes"]

# Conversion to networkx format
G = nx.Graph()
G.add_nodes_from(range(num_nodes))
G.add_edges_from([(edges[0][i], edges[1][i]) for i in range(num_edges)])

# Plot
nx.draw(G)

In [ ]:
len(graph["edge_index"])

In [ ]:
from transformers import GraphormerForGraphClassification

model = GraphormerForGraphClassification.from_pretrained(
    "clefourrier/pcqm4mv2_graphormer_base",
    num_classes=2, # num_classes for the downstream task 
    ignore_mismatched_sizes=True,
)


In [ ]:
model.encoder.graph_encoder

In [ ]:
from transformers.models.graphormer.collating_graphormer import preprocess_item, GraphormerDataCollator

preprocess_item

In [ ]:
print_shapes(preprocess_item(dataset["train"][0]))

In [ ]:
dataset["train"][0].keys()

In [ ]:
dataset["train"][0]["num_nodes"]

In [ ]:
from torch_geometric.datasets import MoleculeNet
dataset = MoleculeNet(root="./", name="ESOL")
dataset

In [ ]:
dataset[0]

In [ ]:
dataset.num_node_features, dataset.num_edge_features

In [ ]:
import torch
import torch_geometric.graphgym.register as register
from torch_geometric.graphgym.config import cfg
from torch_geometric.graphgym.models.gnn import FeatureEncoder, GNNPreMP
from torch_geometric.graphgym.register import register_network

In [ ]:
df = pd.read_csv('../data/sample_submission.csv')

In [ ]:
df["t"] = df["ID"].apply(lambda x: x.split(':')[0])
df["t_1"] = df["ID"].apply(lambda x: x.split(':')[1])
df["t_2"] = df["ID"].apply(lambda x: x.split(':')[2])

In [ ]:
df.query('t == "layout"').groupby(['t_1', 't_2']).count()

In [4]:
import tensorflow_gnn

ModuleNotFoundError: No module named 'tensorflow_gnn'